# Lab: การตรวจจับและแบ่งส่วนวัตถุตามสั่งโดยไม่ต้องเทรนโมเดลด้วย YOLOE-26 (Open-Vocabulary Object Detection & Segmentation)

ยินดีต้อนรับสู่ Lab การใช้งาน **YOLOE-26** ซึ่งเป็นโมเดล Open-Vocabulary (ตรวจจับและแบ่งส่วนวัตถุได้ทันทีจากคำสั่งข้อความหรือภาพอ้างอิงโดยไม่ต้องเทรนโมเดลใหม่) ล่าสุดจากค่าย **Ultralytics**

ใน Lab นี้เราจะพัฒนาโปรแกรมประยุกต์บนระบบปฏิบัติการ Windows/PC โดยใช้กล้องเว็บแคม (Webcam) หรือไฟล์วิดีโอผ่านไลบรารี OpenCV เพื่อแสดงผลลัพธ์แบบเรียลไทม์

### สิ่งที่คุณจะได้เรียนรู้ใน Lab นี้:
1. การติดตั้งไลบรารีที่จำเป็นและการโหลดโมเดลสำเร็จรูป `yoloe-26s-seg.pt`
2. การทำงานในโหมดตรวจจับวัตถุทั่วไปโดยอัตโนมัติ (Prompt-Free Mode)
3. การตรวจจับวัตถุตามสั่งโดยการป้อนข้อความคำอธิบายภาษาอังกฤษ (Custom Text Prompting)
4. การตรวจจับวัตถุตามสั่งโดยการป้อนภาพตัวอย่าง (Visual/Image Prompting) ด้วย `YOLOEVPSegPredictor`
5. การนำผลลัพธ์การตรวจจับมาต่อยอด เช่น การนับจำนวนวัตถุ (Object Counter) และการติดตามตำแหน่ง (Location Tracking)

## 1. ติดตั้งไลบรารีที่จำเป็น (Installation)

เราจำเป็นต้องติดตั้งไลบรารี `ultralytics` รุ่นล่าสุดที่รองรับคลาส `YOLOE` พร้อมกับไลบรารีช่วยทำงานอย่าง `onnxruntime`, `opencv-python`, `numpy` และ `matplotlib`

In [ ]:
# อัปเดตไลบรารี ultralytics และติดตั้งโมดูลที่เกี่ยวข้องสำหรับประมวลผลและการแสดงผล
!pip install -U ultralytics onnxruntime opencv-python numpy matplotlib

## 2. นำเข้าไลบรารีและตรวจสอบคลาส YOLOE

เราจะนำเข้า OpenCV, NumPy, Matplotlib และโหลดคลาส `YOLOE` ของ Ultralytics เข้ามาใช้งานในระบบ

In [2]:
import cv2  # นำเข้าไลบรารี OpenCV สำหรับจัดการกล้อง วิดีโอ และการแสดงผลหน้าต่างภาพ
import numpy as np  # นำเข้า NumPy สำหรับประมวลผลข้อมูลโครงสร้างอาร์เรย์ (เช่น พิกัดกล่อง)
import matplotlib.pyplot as plt  # นำเข้า Matplotlib สำหรับการแสดงผลภาพนิ่งใน Notebook
import os  # นำเข้าโมดูล os สำหรับจัดการกับไฟล์และเส้นทางโฟลเดอร์
from ultralytics import YOLOE  # นำเข้าคลาส YOLOE เพื่อดึงโมเดลสำหรับทำคำสั่งภาษาอังกฤษ/ภาพ
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor  # นำเข้าตัวทำนายสำหรับการส่งภาพอ้างอิง (Visual Prompt)

print("นำเข้าไลบรารีที่จำเป็นทั้งหมดเรียบร้อยแล้ว!")

นำเข้าไลบรารีที่จำเป็นทั้งหมดเรียบร้อยแล้ว!


In [3]:
model_name = "yoloe-26s-seg.pt"  # กำหนดชื่อไฟล์โมเดล YOLOE-26 ขนาดเล็กสำหรับเซกเมนต์
model = YOLOE(model_name)  # โหลดโมเดลเข้ามาในตัวแปร model (ถ้าไม่มีโปรแกรมจะดาวน์โหลดอัตโนมัติ)
print("ดาวน์โหลดและโหลดโมเดล YOLOE สำเร็จ!")

ดาวน์โหลดและโหลดโมเดล YOLOE สำเร็จ!


## 3. การตรวจจับและแบ่งส่วนวัตถุทั่วไปโดยอัตโนมัติ (Prompt-Free Mode)

โหมดนี้เป็นโหมดเบื้องต้นที่โมเดล YOLOE จะตรวจจับวัตถุทั่วไปทั้งหมดที่ตรวจเจอโดยอัตโนมัติ (รองรับวัตถุกว่า 4,800 ชนิด) โดยเราจะเปิดใช้กล้องเว็บแคมผ่าน OpenCV และทำการประมวลผลทีละเฟรม

In [10]:
# ส่วนทดสอบการจับภาพสดจากกล้องเว็บแคม (ID: 0) ในโหมดทั่วไป (Prompt-Free)
# 🌟 โหลดโมเดล YOLOE รุ่น Prompt-Free (สแกนวัตถุทั่วไปโดยอัตโนมัติ) เพื่อใช้งานในโหมดนี้โดยเฉพาะ
pf_model = YOLOE("yoloe-26s-seg-pf.pt")

#cap = cv2.VideoCapture(0)  # เปิดกล้องเว็บแคมหลักผ่านระบบ OpenCV (ปรับหมายเลข 0 เป็นไฟล์วิดีโอเช่น 'sample_video.mp4' ได้หากไม่มีกล้อง)
cap = cv2.VideoCapture('sample_video.mp4')
if not cap.isOpened():  # ตรวจสอบว่าเปิดกล้องเว็บแคมสำเร็จหรือไม่
    print("ไม่สามารถเชื่อมต่อกับกล้องเว็บแคมได้!")
else:
    cv2.namedWindow("YOLOE Prompt-Free Detection", cv2.WINDOW_NORMAL)  # สร้างหน้าต่างแสดงผลที่ยืดหยุ่นขนาดได้
    print("เริ่มตรวจจับวัตถุ กดปุ่ม 'q' หรือ 'Esc' ในหน้าต่างกล้องเพื่อออกจากการทำงาน")
    
    while True:
        ret, frame = cap.read()  # อ่านเฟรมล่าสุดจากกล้อง (ret เป็น True หากมีเฟรมเข้ามา)
        if not ret:  # หากอ่านเฟรมไม่สำเร็จ ให้หยุดการทำซ้ำ
            break
            
        results = pf_model.predict(frame, verbose=False)  # 🌟 ใช้ pf_model ทำนาย (มีคลาสกว่า 1,200+ ชนิดในตัว)
        annotated_frame = results[0].plot(boxes=True, masks=True)  # วาดกล่อง (boxes) และหน้ากากแบ่งส่วน (masks) ลงบนภาพ
        
        # แสดงผลภาพที่ผ่านการระบุวัตถุลงบนหน้าต่างกล้อง
        cv2.imshow("YOLOE Prompt-Free Detection", annotated_frame)
        
        key = cv2.waitKey(1) & 0xFF  # เช็กการกดปุ่มบนคีย์บอร์ด
        if key == ord('q') or key == 27:  # ถ้ากดปุ่ม 'q' หรือปุ่ม Esc (27) ให้หยุดการทำงาน
            break
            
    cap.release()  # ปล่อยการเชื่อมต่อกล้องเว็บแคมเพื่อประหยัดทรัพยากร
    cv2.destroyAllWindows()  # ปิดหน้าต่างแสดงผลของ OpenCV ทั้งหมด
    print("ปิดการใช้งานกล้องเว็บแคมเรียบร้อยแล้ว")

เริ่มตรวจจับวัตถุ กดปุ่ม 'q' หรือ 'Esc' ในหน้าต่างกล้องเพื่อออกจากการทำงาน
ปิดการใช้งานกล้องเว็บแคมเรียบร้อยแล้ว


## 4. การตรวจจับวัตถุตามสั่งด้วยคำค้นหา (Custom Text Prompting)

จุดเด่นของ YOLOE คือการเป็น Open-Vocabulary Model ซึ่งช่วยให้เราสามารถพิมพ์คำค้นหาเป็นภาษาอังกฤษใดๆ ก็ได้เพื่อกำหนดวัตถุที่ต้องการจับภาพแบบไดนามิกโดยใช้เมธอด `set_classes` ในทันที

In [15]:
# ตั้งค่าคลาสที่คุณอยากตรวจจับในรูปแบบข้อความ (Text Prompts) โดยไม่ต้องทำการเทรนใหม่
prompts = ["coffee cup", "remote", "keyboard", "human", "key" , "face"]  # กำหนดคำภาษาอังกฤษของสิ่งของที่ต้องการจับ
model.set_classes(prompts)  # ป้อนคำอธิบายเหล่านี้ให้ระบบโมเดลเข้าใจ
print(f"ตั้งค่าคลาสตรวจจับเรียบร้อย: {prompts}")

cap = cv2.VideoCapture(0)  # เปิดกล้องเว็บแคมเว็บแคมอีกครั้ง
if not cap.isOpened():
    print("ไม่สามารถเชื่อมต่อกับกล้องเว็บแคมได้!")
else:
    cv2.namedWindow("YOLOE Text Prompt Detection", cv2.WINDOW_NORMAL)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        results = model.predict(frame, verbose=False)  # ทำนายเฟรมภาพโดยค้นหาเฉพาะเป้าหมายใน prompts
        annotated_frame = results[0].plot(boxes=True, masks=True)  # วาดกล่องและไฮไลต์หน้ากากแบ่งส่วน
        
        cv2.imshow("YOLOE Text Prompt Detection", annotated_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break
            
    cap.release()
    cv2.destroyAllWindows()

ตั้งค่าคลาสตรวจจับเรียบร้อย: ['coffee cup', 'remote', 'keyboard', 'human', 'key', 'face']


In [21]:
# ตั้งค่าคลาสที่คุณอยากตรวจจับในรูปแบบข้อความ (Text Prompts) โดยไม่ต้องทำการเทรนใหม่
prompts = ["coffee cup", "remote", "keyboard", "human face"]
model.set_classes(prompts)
print(f"ตั้งค่าคลาสตรวจจับเรียบร้อย: {prompts}")
cap = cv2.VideoCapture(0)  # เปิดกล้องเว็บแคม
if not cap.isOpened():
    print("ไม่สามารถเชื่อมต่อกับกล้องเว็บแคมได้!")
else:
    # 🌟 [เพิ่ม] บังคับความละเอียดของกล้องเป็น 1280x720 (HD) เพื่อความคมชัดในระยะไกล
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    cv2.namedWindow("YOLOE Text Prompt Detection", cv2.WINDOW_NORMAL)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # 🌟 [ปรับปรุง] เพิ่ม imgsz=1280 (ตรวจจับรายละเอียดเล็กๆ) และ conf=0.15 (เพิ่มความไวการจับวัตถุไกล)
        results = model.predict(frame, imgsz=1280, conf=0.09, verbose=False)
        annotated_frame = results[0].plot(boxes=True, masks=True)
        
        cv2.imshow("YOLOE Text Prompt Detection", annotated_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break
            
    cap.release()
    cv2.destroyAllWindows()

ตั้งค่าคลาสตรวจจับเรียบร้อย: ['coffee cup', 'remote', 'keyboard', 'human face']


## 5. การตรวจจับวัตถุด้วยภาพตัวอย่าง (YOLOE Visual Prompting)

สำหรับวัตถุที่เป็นเอกลักษณ์หรือมีชื่อยากๆ ที่ระบบไม่เข้าใจจากข้อความ YOLOE สนับสนุนการใช้ภาพอ้างอิง (Visual/Image Prompting) โดยเราจะถ่ายภาพวัตถุไว้เป็นตัวอย่าง (`ref.jpg`) และส่งพิกัดกล่องของตัวอย่างให้โมเดลค้นหาวัตถุแบบเดียวกัน

In [8]:
# ส่วนการทำ Visual Prompting (การใช้รูปตัวอย่างในการตามหาวัตถุ)
# 1. ถ่ายภาพหรือดึงภาพที่มีวัตถุเพื่อใช้เป็นภาพอ้างอิง (Reference)
# ในขั้นตอนการเขียนโปรแกรม เราจะดึงภาพจากกล้องเว็บแคมมาบันทึกเป็น ref.jpg เพื่อดึงกล่องพิกัดมาสอนโมเดล

cap = cv2.VideoCapture(1)
if not cap.isOpened():
    print("ไม่สามารถเชื่อมต่อกับกล้องเว็บแคมได้!")
else:
    print("จัดวางวัตถุไว้หน้ากล้อง แล้วกดปุ่ม Spacebar เพื่อบันทึกภาพอ้างอิง")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow("Press Space to capture Reference Image", frame)
        if cv2.waitKey(1) & 0xFF == 32:  # 32 คือปุ่ม Spacebar
            cv2.imwrite("ref.jpg", frame)  # บันทึกภาพอ้างอิงลงเครื่องเป็นไฟล์ ref.jpg
            print("บันทึกภาพ ref.jpg สำเร็จ!")
            break
    cap.release()
    cv2.destroyAllWindows()

# 2. จำลองการเลือกพิกัดสี่เหลี่ยมรอบวัตถุในภาพอ้างอิง (Bounding Box)
# สมมติพิกัดเป็น [x1, y1, x2, y2]
ref_img = cv2.imread("ref.jpg")
if ref_img is not None:
    h, w, _ = ref_img.shape
    # กำหนดจุดกึ่งกลางของภาพอ้างอิงเพื่อเป็นตัวอย่างกล่องพิกัดเริ่มต้นในการคำนวณ
    box_coords = [int(w*0.25), int(h*0.25), int(w*0.75), int(h*0.75)]
    print(f"พิกัดกล่องอ้างอิงเบื้องต้น: {box_coords} (กว้าง x สูง: {w}x{h})")

    # 3. ส่งข้อมูลรูปอ้างอิงและกล่องวัตถุให้ YOLOEVPSegPredictor ค้นหาในภาพจริงสดๆ จากกล้อง
    visual_prompts = dict(
        bboxes=np.array([box_coords]),  # พิกัดกล่องของวัตถุบนภาพอ้างอิง
        cls=np.array([0])  # กำหนดให้เป็นวัตถุคลาสหลัก (หมายเลข 0)
    )

    cap = cv2.VideoCapture(0)
    if cap.isOpened():
        cv2.namedWindow("YOLOE Visual Prompt Detection", cv2.WINDOW_NORMAL)
        print("กำลังทำงานในโหมดตรวจจับด้วยภาพตัวอย่าง...")
        while True:
            ret, frame = cap.read()
            if not ret:
                break
                
            # ส่ง target เป็นเฟรมกล้อง, refer_image เป็นภาพอ้างอิง, visual_prompts เป็นจุดพิกัดวัตถุในภาพอ้างอิง
            results = model.predict(
                source=frame,
                refer_image="ref.jpg",
                visual_prompts=visual_prompts,
                predictor=YOLOEVPSegPredictor,
                verbose=False
            )
            
            annotated_frame = results[0].plot(boxes=True, masks=True)
            cv2.imshow("YOLOE Visual Prompt Detection", annotated_frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q') or key == 27:
                break
        cap.release()
        cv2.destroyAllWindows()
else:
    print("ไม่พบไฟล์ ref.jpg")

## 6. การประยุกต์ใช้งานจริง (Counting & Location Tracking)

ในส่วนนี้เราจะนำพิกัดการตรวจจับและผลความมั่นใจของโมเดลมานับจำนวน (Counter) รวมถึงดึงพิกัดกึ่งกลาง X, Y (ปรับสเกลเป็น 0.0 - 1.0) เพื่อตรวจสอบว่าวัตถุอยู่ในส่วนใดของจอภาพ เช่น หากมาอยู่บริเวณมุมล่างขวา ให้ทำการกระตุ้นการทำงาน (Trigger) บางอย่าง

In [7]:
# ส่วนการนับจำนวนวัตถุและการตรวจสอบตำแหน่งพิกัด X, Y บนหน้าจอ
TARGET_OBJECT = "coffee cup"  # กำหนดชนิดวัตถุที่เราเพิ่งตั้งค่าไว้ใน Text Prompts
CONFIDENCE_THRESHOLD = 0.25  # กำหนดระดับความมั่นใจขั้นต่ำที่ยอมรับได้ (0.0 ถึง 1.0)

# คืนค่าโมเดลกลับมาใช้ Text Prompts ของเรา
model.set_classes([TARGET_OBJECT])

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("ไม่สามารถเชื่อมต่อกับกล้องเว็บแคมได้!")
else:
    cv2.namedWindow("YOLOE Counter & Location Tracking", cv2.WINDOW_NORMAL)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        results = model.predict(frame, verbose=False)
        annotated_frame = results[0].plot(boxes=True, masks=False)
        
        # ตรวจสอบผลลัพธ์การตรวจจับ
        boxes = results[0].boxes
        object_count = 0
        
        for box in boxes:
            confidence = float(box.conf[0])  # ดึงค่าความมั่นใจของกล่องนั้นๆ
            if confidence >= CONFIDENCE_THRESHOLD:
                object_count += 1  # นับจำนวนวัตถุที่ผ่านเกณฑ์
                
                # พิกัดกล่องแบบมุม: [x1, y1, x2, y2]
                x1, y1, x2, y2 = box.xyxy[0]
                # คำนวณหาจุดศูนย์กลางของวัตถุเทียบเป็นสัดส่วน (0.0 ถึง 1.0)
                cx = float(((x1 + x2) / 2) / frame.shape[1])
                cy = float(((y1 + y2) / 2) / frame.shape[0])
                
                # แสดงพิกัด X, Y ในหน้าต่าง Console
                print(f"พบ {TARGET_OBJECT}! พิกัดกึ่งกลาง: X={cx:.2f}, Y={cy:.2f} (ความมั่นใจ: {confidence:.2f})")
                
                # ตัวอย่างเงื่อนไข: หากอยู่บริเวณมุมล่างขวา (X > 0.5 และ Y > 0.5)
                if cx > 0.5 and cy > 0.5:
                    cv2.putText(annotated_frame, "TRIGGER: BOTTOM RIGHT!", (10, 80), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
        # พิมพ์และแสดงจำนวนวัตถุบนภาพ
        count_text = f"Count: {object_count}"
        cv2.putText(annotated_frame, count_text, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        cv2.imshow("YOLOE Counter & Location Tracking", annotated_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break
            
    cap.release()
    cv2.destroyAllWindows()
    print("ปิดหน้าต่างตรวจจับเรียบร้อยแล้ว")

ปิดหน้าต่างตรวจจับเรียบร้อยแล้ว


## 6. การประเมินประสิทธิภาพด้วยชุดข้อมูลทดสอบ (Validation)

นอกจากการดูผลลัพธ์ผ่านกล้องสดแล้ว เรายังสามารถประเมินความแม่นยำของโมเดลอย่างเป็นระบบด้วยชุดข้อมูลมาตรฐาน (เช่น Dataset ในรูปแบบ COCO format)

ในเซลล์นี้ เราจะใช้ชุดข้อมูลทดสอบจิ๋วสำเร็จรูปของ Ultralytics ที่ชื่อว่า `coco8-seg.yaml` (มีภาพทั้งหมด 8 ภาพสำหรับการทดสอบ) โดยเรียกใช้งานเมธอด `val()` ของ YOLOE เพื่อประเมินค่า **mAP (Mean Average Precision)** ทั้งสำหรับการตรวจจับแบบกล่อง (Box) และการแบ่งส่วนภาพ (Segmentation)

In [ ]:
# รันการวัดประสิทธิภาพของโมเดลด้วยชุดข้อมูล coco8-seg
# หมายเหตุ: ใน Windows ควรรันภายใต้เงื่อนไข `if __name__ == '__main__':` เสมอ เพื่อป้องกันปัญหา multiprocessing หลูปซ้ำ

def run_validation():
    print("กำลังเริ่มต้นการประเมินประสิทธิภาพบน dataset 'coco8-seg'...")
    # รันการวัดผล (กำหนด workers=0 บน Windows เพื่อหลีกเลี่ยงปัญหาของตัวโหลดข้อมูล)
    metrics = model.val(data="coco8-seg.yaml", workers=0)
    
    print("\n=== ผลการประเมินประสิทธิภาพ (Validation Results) ===")
    print(f"mAP50-95 (Box): {metrics.box.map:.4f} (ความแม่นยำเฉลี่ยของกรอบตรวจจับวัตถุ)")
    print(f"mAP50 (Box):    {metrics.box.map50:.4f} (ความแม่นยำที่เกณฑ์ความทับซ้อน 50%)")
    print(f"mAP50-95 (Seg): {metrics.seg.map:.4f} (ความแม่นยำเฉลี่ยของการแบ่งส่วนภาพ Mask)")
    print(f"mAP50 (Seg):    {metrics.seg.map50:.4f} (ความแม่นยำการแบ่งส่วนที่เกณฑ์ความทับซ้อน 50%)")

if __name__ == '__main__':
    run_validation()

กำลังเริ่มต้นการประเมินประสิทธิภาพบน dataset 'coco8-seg'...
Validate using the text prompt.
Ultralytics 8.4.82  Python-3.10.11 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 635.6317.3 MB/s, size: 54.0 KB)
val: Scanning E:\test\datasets\coco8-seg\labels\val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 9.6it/s 0.1s
                   all          4         17      0.864      0.424      0.724      0.441      0.891      0.439      0.724      0.433
                person          3         10      0.843      0.542      0.695      0.405          1      0.636      0.695      0.274
                   dog          1          1          1          0      0.995     0.0995          1          0      0.995      0.398
                 horse    

## 7. ระบบวิเคราะห์และนับจำนวนรถยนต์แยกเลน ซ้าย (OUT) / ขวา (IN) (Vehicle Counting & Tracking)

ในส่วนนี้เราจะนำฟังก์ชันการติดตามวัตถุ (Object Tracking) ของ YOLOE มาต่อยอดพัฒนาเป็นระบบนับรถยนต์แยกทิศทาง โดยแบ่งออกเป็น:
- **เลนซ้าย (Outbound / OUT):** รถที่วิ่งออก (ทิศทางวิ่งขึ้นด้านบนจอ - ค่า Y ลดลง)
- **เลนขวา (Inbound / IN):** รถที่วิ่งเข้าหา (ทิศทางวิ่งลงด้านล่างจอ - ค่า Y เพิ่มขึ้น)

เราจะนำไฟล์วิดีโอตัวอย่าง `Road-traffic-video_2m.mp4` มาตรวจจับ สรรสร้างเส้นตรวจสอบเสมือนจริง (Virtual Line) และแสดงจำนวนรถที่ผ่านสะสมบนหน้าจอแบบเรียลไทม์ผ่าน OpenCV

In [80]:
# โค้ดระบบนับรถยนต์แยกเลน OUT (ซ้าย) / IN (ขวา) ด้วย YOLOE + OpenCV (ปรับปรุงให้แม่นยำยิ่งขึ้น)

def run_vehicle_counter():
    model = YOLOE("yoloe-26s-seg.pt")
    model.set_classes(["car", "truck", "bus"])
    
    cap = cv2.VideoCapture(r"f:\\test\\yoloe_lab\\Road-traffic-video_2m.mp4")
    if not cap.isOpened():
        print("เปิดวิดีโอไม่ได้!"); return
    
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"วิดีโอ: {w}x{h} @ {fps} FPS")
    
    LINE_Y, DIV_X = 500, 660  # เส้นนับ (แนวขวาง) และจุดแบ่งเลน (เกาะกลางถนน)
    out_count, in_count = 0, 0
    counted = set()          # เซ็ตเก็บข้อมูลที่เคยนับแล้วเพื่อป้องกันการนับซ้ำ: {('L', tid), ('R', tid)}
    
    # 🌟 [ปรับปรุง] ใช้พจนานุกรมเก็บประวัติพิกัดจุดกึ่งกลางย้อนหลังสูงสุด 15 เฟรม เพื่อใช้ตรวจจับการข้ามเส้นที่เสถียรขึ้น
    # โครงสร้าง: {tid: [(cx1, cy1), (cx2, cy2), ...]}
    track_history = {}
    track_last_seen = {}     # เก็บเฟรมล่าสุดที่พบ ID นี้เพื่อล้างข้อมูลที่ไม่ได้ใช้งานออก
    frame_idx = 0
    
    cv2.namedWindow("YOLOE Lane Vehicle Counter (Improved)", cv2.WINDOW_NORMAL)
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        frame_idx += 1
        
        # 🌟 [ปรับปรุง] ใช้ imgsz=960 และ conf=0.15 เพื่อให้ตรวจจับได้เสถียรขึ้นและลดการสลับ ID คืน
        results = model.track(frame, imgsz=960, conf=0.15, persist=True, verbose=False)
        annotated = results[0].plot(boxes=True, masks=False)
        
        # วาดเส้นนับ: แดง (OUT) | น้ำเงิน (IN)
        cv2.line(annotated, (0, LINE_Y), (DIV_X, LINE_Y), (0, 0, 255), 3)
        cv2.line(annotated, (DIV_X, LINE_Y), (w, LINE_Y), (255, 0, 0), 3)
        
        boxes = results[0].boxes
        if boxes is not None and boxes.id is not None:
            tids = boxes.id.int().tolist()
            xyxy_list = boxes.xyxy.tolist()
            
            for tid, (x1, y1, x2, y2) in zip(tids, xyxy_list):
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)
                
                if tid not in track_history:
                    track_history[tid] = []
                track_history[tid].append((cx, cy))
                
                # จำกัดขนาดของประวัติให้เก็บย้อนหลังไม่เกิน 15 เฟรมเพื่อไม่ให้ใช้หน่วยความจำมากเกินไป
                if len(track_history[tid]) > 15:
                    track_history[tid].pop(0)
                
                track_last_seen[tid] = frame_idx
                
                # 🌟 [ปรับปรุง] วาดเส้นประวัติการเคลื่อนที่ (Trail) ของรถแต่ละคันเพื่อช่วยวิเคราะห์ตำแหน่งวิ่ง
                for i in range(1, len(track_history[tid])):
                    cv2.line(annotated, track_history[tid][i-1], track_history[tid][i], (0, 255, 255), 2)
                cv2.circle(annotated, (cx, cy), 5, (0, 255, 255), -1)
                
                # 🌟 [ปรับปรุง] ระบบวิเคราะห์การข้ามเส้นโดยอิงประวัติหลายเฟรม (Trajectory Crossover Detection)
                if cx < DIV_X:
                    # เลนซ้าย (OUT): รถวิ่งขึ้นด้านบนจอ (Y ลดลง) ข้ามจากล่างขึ้นบน
                    if cy < LINE_Y and ('L', tid) not in counted:
                        has_crossed_up = any(pt[1] >= LINE_Y for pt in track_history[tid])
                        if has_crossed_up:
                            counted.add(('L', tid))
                            out_count += 1
                else:
                    # เลนขวา (IN): รถวิ่งลงเข้าหากล้อง (Y เพิ่มขึ้น) ข้ามจากบนลงล่าง
                    if cy > LINE_Y and ('R', tid) not in counted:
                        has_crossed_down = any(pt[1] <= LINE_Y for pt in track_history[tid])
                        if has_crossed_down:
                            counted.add(('R', tid))
                            in_count += 1
        
        # 🌟 [ปรับปรุง] ล้างข้อมูลประวัติของ ID ที่หลุดออกไปจากหน้าจอเพื่อป้องกัน memory leak
        for tid in list(track_history.keys()):
            if frame_idx - track_last_seen[tid] > 30:
                del track_history[tid]
                del track_last_seen[tid]
        
        # แสดงสถิติสะสมบนหน้าจอ
        cv2.putText(annotated, f"OUT (Left): {out_count}", (50, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        cv2.putText(annotated, f"IN (Right): {in_count}", (w - 400, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)
        
        cv2.imshow("YOLOE Lane Vehicle Counter", annotated)
        if cv2.waitKey(1) & 0xFF in (ord('q'), 27): break
    
    cap.release()
    cv2.destroyAllWindows()
    print(f"วิเคราะห์วิดีโอจราจรเสร็จสิ้น! ยอดนับรถได้: OUT={out_count} คัน, IN={in_count} คัน")

if __name__ == '__main__':
    run_vehicle_counter()


วิดีโอ: 1280x720 @ 25.0 FPS
วิเคราะห์วิดีโอจราจรเสร็จสิ้น! ยอดนับรถได้: OUT=5 คัน, IN=4 คัน


## 8. ระบบวิเคราะห์และนับจำนวนรถยนต์แยกเลน ซ้าย (OUT) / ขวา (IN) ด้วยเส้นตรวจจับต่างระนาบและต่างสี (Lane Vehicle Counter with Offset Lines & Colors)

ในส่วนนี้เราจะยกระดับระบบวิเคราะห์และนับจำนวนรถยนต์ไปอีกขั้น โดยเพิ่มความยืดหยุ่นและการทำงานแยกอิสระดังนี้:
- **เลนซ้าย (Outbound / รถขาออก):** รถที่วิ่งออก (ทิศทางเคลื่อนที่ขึ้นด้านบนจอ) เมื่อข้าม **เส้นตรวจจับสีเขียว (Green Line)** จะทำการนับสะสม
- **เลนขวา (Inbound / รถขาเข้า):** รถที่วิ่งเข้าหา (ทิศทางเคลื่อนที่ลงด้านล่างจอ) เมื่อข้าม **เส้นตรวจจับสีแดง (Red Line)** จะทำการนับสะสม
- **เส้นตรวจจับต่างระนาบ (Offset Lines):** เพื่อจำลองสถานการณ์จริงที่กล้องจราจรมองทำมุมเฉียง และความเร็วหรือจุดตรวจจับที่เหมาะสมของแต่ละเลนไม่เท่ากัน เราจึงตั้งค่าระดับแนวขวาง Y แยกกัน คือ `LINE_L_Y` สำหรับเลนซ้าย และ `LINE_R_Y` สำหรับเลนขวา โดยไม่บังคับให้อยู่ในระนาบเดียวกัน
- **การแสดงผล:** แสดงการวิเคราะห์แบบเรียลไทม์ผ่านวิดีโอ `Road-traffic-video_2m.mp4` วาดกล่องตรวจจับ, เส้นตรวจจับสองสีแยกฝั่ง, และวาดเส้นวิถีการเคลื่อนที่ (Trail) สีเหลืองของรถแต่ละคัน

In [81]:
# โค้ดระบบนับรถยนต์แยกเลน ซ้าย (OUT - เส้นเขียว) / ขวา (IN - เส้นแดง) ด้วยเส้นตรวจจับต่างระดับ

def run_vehicle_counter_offset_lines():
    # โหลดโมเดล YOLOE และกำหนดคลาสที่ต้องการตรวจจับ (รถยนต์, รถบรรทุก, รถบัส)
    model = YOLOE("yoloe-26s-seg.pt")
    model.set_classes(["car", "truck", "bus"])
    
    # โหลดไฟล์วิดีโอ
    cap = cv2.VideoCapture(r"f:\\test\\yoloe_lab\\Road-traffic-video_2m.mp4")
    if not cap.isOpened():
        print("ไม่สามารถเปิดวิดีโอได้!")
        return
    
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"วิดีโอ: {w}x{h} @ {fps} FPS")
    
    # กำหนดเส้นนับ (แนวขวาง) และจุดแบ่งเลน (เกาะกลางถนน)
    LINE_L_Y = 460  # เส้นตรวจจับเลนซ้าย (ขาออก) - อยู่สูงขึ้นไป (ค่า Y น้อยกว่า)
    LINE_R_Y = 530  # เส้นตรวจจับเลนขวา (ขาเข้า) - อยู่ต่ำลงมา (ค่า Y มากกว่า)
    DIV_X = 660     # จุดกึ่งกลางแบ่งถนนซ้าย/ขวา
    
    # กำหนดสีสำหรับวาดลงเฟรม (BGR)
    COLOR_GREEN = (0, 255, 0)     # สีเขียวสำหรับเส้นซ้าย
    COLOR_RED = (0, 0, 255)       # สีแดงสำหรับเส้นขวา
    COLOR_YELLOW = (0, 255, 255)  # สีเหลืองสำหรับเส้นประวัติเคลื่อนที่ (Trail)
    
    out_count, in_count = 0, 0
    counted = set()          # เซ็ตเก็บข้อมูลที่เคยนับแล้วเพื่อป้องกันการนับซ้ำ: {('L', tid), ('R', tid)}
    
    # พจนานุกรมเก็บประวัติพิกัดจุดกึ่งกลางย้อนหลังสูงสุด 15 เฟรม
    track_history = {}
    track_last_seen = {}
    frame_idx = 0
    
    cv2.namedWindow("YOLOE Offset Lines Vehicle Counter", cv2.WINDOW_NORMAL)
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_idx += 1
        
        # ใช้ imgsz=960 และ conf=0.15 เพื่อการจับวัตถุและติดตาม (Tracking) ที่มีความเสถียรสูงสุด
        results = model.track(frame, imgsz=960, conf=0.15, persist=True, verbose=False)
        annotated = results[0].plot(boxes=True, masks=False)
        
        # วาดเส้นตรวจจับต่างระดับและต่างสี
        # เลนซ้าย (ขาออก): เส้นสีเขียว
        cv2.line(annotated, (0, LINE_L_Y), (DIV_X, LINE_L_Y), COLOR_GREEN, 3)
        # เลนขวา (ขาเข้า): เส้นสีแดง
        cv2.line(annotated, (DIV_X, LINE_R_Y), (w, LINE_R_Y), COLOR_RED, 3)
        
        # วาดข้อความกำกับแต่ละเส้น
        cv2.putText(annotated, "GREEN LINE (Outbound)", (10, LINE_L_Y - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_GREEN, 2)
        cv2.putText(annotated, "RED LINE (Inbound)", (DIV_X + 10, LINE_R_Y - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_RED, 2)
        
        boxes = results[0].boxes
        if boxes is not None and boxes.id is not None:
            tids = boxes.id.int().tolist()
            xyxy_list = boxes.xyxy.tolist()
            
            for tid, (x1, y1, x2, y2) in zip(tids, xyxy_list):
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)
                
                if tid not in track_history:
                    track_history[tid] = []
                track_history[tid].append((cx, cy))
                
                if len(track_history[tid]) > 15:
                    track_history[tid].pop(0)
                
                track_last_seen[tid] = frame_idx
                
                # วาดเส้นประวัติการเคลื่อนที่ (Trail) ของรถแต่ละคันเป็นสีเหลือง
                for i in range(1, len(track_history[tid])):
                    cv2.line(annotated, track_history[tid][i-1], track_history[tid][i], COLOR_YELLOW, 2)
                cv2.circle(annotated, (cx, cy), 5, COLOR_YELLOW, -1)
                
                # ระบบวิเคราะห์การข้ามเส้นโดยอิงทิศทางและตำแหน่งตามเลน
                if cx < DIV_X:
                    # เลนซ้าย (ขาออก / OUT): วิ่งขึ้น (Y ลดลง) ข้ามผ่านเส้นสีเขียว (LINE_L_Y)
                    if cy < LINE_L_Y and ('L', tid) not in counted:
                        has_crossed_up = any(pt[1] >= LINE_L_Y for pt in track_history[tid])
                        if has_crossed_up:
                            counted.add(('L', tid))
                            out_count += 1
                else:
                    # เลนขวา (ขาเข้า / IN): วิ่งลง (Y เพิ่มขึ้น) ข้ามผ่านเส้นสีแดง (LINE_R_Y)
                    if cy > LINE_R_Y and ('R', tid) not in counted:
                        has_crossed_down = any(pt[1] <= LINE_R_Y for pt in track_history[tid])
                        if has_crossed_down:
                            counted.add(('R', tid))
                            in_count += 1
        
        # ล้างข้อมูลประวัติคันที่ไม่ได้อยู่ในจอแล้วเพื่อลดการใช้หน่วยความจำ
        for tid in list(track_history.keys()):
            if frame_idx - track_last_seen[tid] > 30:
                del track_history[tid]
                del track_last_seen[tid]
        
        # แสดงสถิติสะสมแบบแยกทิศทาง
        cv2.putText(annotated, f"OUT (Left - Green): {out_count}", (50, 60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, COLOR_GREEN, 3)
        cv2.putText(annotated, f"IN (Right - Red): {in_count}", (w - 550, 60), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, COLOR_RED, 3)
        
        cv2.imshow("YOLOE Offset Lines Vehicle Counter", annotated)
        if cv2.waitKey(1) & 0xFF in (ord('q'), 27):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print(f"วิเคราะห์วิดีโอจราจรเสร็จสิ้น! ยอดนับรถได้: OUT={out_count} คัน, IN={in_count} คัน")

if __name__ == '__main__':
    run_vehicle_counter_offset_lines()


วิดีโอ: 1280x720 @ 25.0 FPS
วิเคราะห์วิดีโอจราจรเสร็จสิ้น! ยอดนับรถได้: OUT=40 คัน, IN=16 คัน
